In [1]:
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
import openai

In [3]:
load_dotenv()
client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# 1. עמודות בסיס קיימות
speakers = ["patient", "caregiver"]
lengths = ["short", "medium", "long"]

# 2. הגדרת פרוטוקול מנצ'סטר (Manchester Triage System) - משתנה המטרה החדש!
triage_levels = [0, 1, 2, 3]

triage_descriptions = {
    0: "Routine Recovery: Standard post-op symptoms (mild, expected soreness, light pinkish discharge, feeling tired but stable). No danger signs.",
    1: "Low Urgency (Mild Complications): Non-urgent issues that need guidance but are not immediate threats (e.g., mild localized redness around stitches without fever, slight constipation, minor baby blues, mild pain managed by pills).",
    2: "Medium Urgency (Urgent Evaluation): Significant symptoms requiring prompt medical assessment within hours (e.g., widening redness or foul discharge from incision, new fever under 38.5°C, inability to void urine, severe persistent headache, moderate burning during urination).",
    3: "High Urgency / Critical (Life-Threatening): Immediate emergency requiring resuscitation or ER triage (e.g., massive vaginal bleeding soaking a pad within 30-60 mins, passing blood clots larger than a golf ball, sudden severe shortness of breath, sharp stabbing chest pain, high fever over 39°C with chills, signs of severe DVT like a completely swollen, hot, purple calf)."
}

# 3. מילון קטגוריות קליניות מגוון (בשביל ה-Multi-label שיישמר ברקע כעמודות)
clinical_categories = [
    "wound_problem", "infection_warning", "bleeding_warning", 
    "respiratory_warning", "severe_pain", "thromboembolism_warning", 
    "mood_disorder", "urinary_problem"
]

# 4. מאגרי גיוון לשוני להזרקה דינמית לפרומפט (הסוד למניעת תבניות קבועות)
patient_backgrounds = [
    "anxious first-time mother who is easily panicked",
    "exhausted mother of three who minimizes her own pain",
    "articulate healthcare worker using technical terms",
    "young woman using casual internet slang and texting style",
    "stressed partner who doesn't know clinical terms",
    "someone writing in a hurry with fragmented thoughts",
    "older mother (over 40) who is highly focused on exact medical measurements", 
    "non-native English speaker making minor grammatical slips but clear in meaning" 
]

# הרחבת הסגנונות ל-6 (מה שיוצר 15 שילובים שונים של זוגות סגנון!)
linguistic_styles = [
    "Use everyday, non-medical language (e.g., 'tummy hurt', 'gushing blood', 'leaking stuff').",
    "Include common medical abbreviations and text shorthand (e.g., C-sec, OBGYN, ER, hrs, mg, w/, bc, tmi, idk, painful upper leg instead of DVT).",
    "Write with realistic human emotion: varying between high panic, frustration, exhaustion, or calm inquiry.",
    "Make the phrasing highly conversational, avoiding rigid structures. Use short, disjointed sentences or run-on sentences.",
    "Include specific timelines in the text (e.g., 'since last night', 'for 3 hours now', 'it started 20 mins ago').", 
    "Express confusion or skepticism, asking if this is normal or if they should worry." 
]

# 5. פונקציה לבניית הפרומפט הדינמי והמורכב
def generate_patient_case_prompt():
    # הגרלת רמת הדחיפות לפי מנצ'סטר
    selected_triage = random.choice(triage_levels)
    
    # אתחול עמודות הקטגוריות
    complications = {cat: 0 for cat in clinical_categories}
    
    selected_speaker = random.choice(speakers)
    selected_length = random.choice(lengths)
    selected_background = random.choice(patient_backgrounds)
    selected_styles = random.sample(linguistic_styles, 2)  # הגרלת 2 חוקי סגנון שונים לכל הרצה
    
    length_instruction = {
        "short": "very brief, around 1-2 sentences max",
        "medium": "moderate length, around 3-5 sentences",
        "long": "detailed and complex, around 6-10 sentences with backstory context"
    }[selected_length]
    
    if selected_speaker == "caregiver":
        speaker_instruction = "The message must be written by a caregiver (e.g., husband, partner, mother) describing the patient's condition. Use third-person (e.g., 'My partner...', 'She...')."
    else:
        speaker_instruction = "The message must be written directly by the patient herself. Use first-person (e.g., 'I...', 'My incision...')."
        
    # הגדרת המצב הרפואי והדלקת קטגוריות רלוונטיות בעמודות
    medical_condition = triage_descriptions[selected_triage]
    
    if selected_triage > 0:
        # אם יש בעיה, נגריל אילו קטגוריות רפואיות באות לידי ביטוי בטקסט
        num_complications = random.choice([1, 2])
        chosen_cats = random.sample(clinical_categories, num_complications)
        for cat in chosen_cats:
            complications[cat] = 1

    system_prompt = "You are an expert medical NLP data generator. Return ONLY a valid JSON object. No markdown, no prose outside JSON."
    
    user_prompt = f"""
    Generate a highly realistic, raw message sent to a clinic/doctor regarding a patient after a C-section.
    
    [WRITER PROFILE]: {speaker_instruction} Act as a {selected_background}.
    [CLINICAL SEVERITY (Manchester Triage Level {selected_triage})]: The text MUST accurately portray symptoms matching this clinical state: {medical_condition}.
    [TEXT LENGTH]: {length_instruction}.
    [LINGUISTIC RULES]:
    - {selected_styles[0]}
    - {selected_styles[1]}
    - DO NOT use predictable textbook phrases like 'I am experiencing a thromboembolism warning'. Use organic, messy human descriptions.
    
    Return a JSON object with exactly these keys:
    1. "patient_message": The natural text in English.
    2. "evidence": A strict verbatim quote from the text that justifies the triage level (leave empty "" for Level 0).
    """
    
    return system_prompt, user_prompt, complications, selected_triage, selected_speaker, selected_length

# 6. לולאת הגנרציה ושמירה בזמן אמת ל-CSV ול-JSON
def build_dataset(num_samples=3000): 
    records = []
    csv_filename = "manchester_postpartum_triage_v1.csv"
    
    # מבנה העמודות החדש - השתנה משתנה המטרה מ-worsening ל-triage_level
    base_columns = ["case_id", "patient_message", "sender", "length_target"]
    cat_columns = [f"cat_{cat}" for cat in clinical_categories]
    metrics_columns = ["triage_level", "char_count", "word_count", "evidence"]
    final_columns = base_columns + cat_columns + metrics_columns
    
    print(f"🚨 Starting generation of {num_samples} samples based on Manchester Triage Protocol...")
    
    # יצירת הקובץ הריק
    pd.DataFrame(columns=final_columns).to_csv(csv_filename, index=False)
    
    for i in range(num_samples):
        sys_p, usr_p, complications_dict, triage_label, speaker, length_target = generate_patient_case_prompt()
        
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                temperature=1.3,  # טמפרטורה גבוהה ליצירתיות וגיוון בשפה
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": sys_p},
                    {"role": "user", "content": usr_p}
                ]
            )
            
            result_json = json.loads(response.choices[0].message.content.strip())
            message_text = result_json["patient_message"]
            
            record = {
                "case_id": i + 1,
                "patient_message": message_text,
                "sender": speaker,
                "length_target": length_target,
                "triage_level": triage_label,
                "char_count": len(message_text),
                "word_count": len(message_text.split()),
                "evidence": result_json["evidence"]
            }
            
            for cat in clinical_categories:
                record[f"cat_{cat}"] = complications_dict[cat]
                
            records.append(record)
            
            # שמירה מיידית של השורה הנוכחית ל-CSV
            df_single_row = pd.DataFrame([record])[final_columns]
            df_single_row.to_csv(csv_filename, mode='a', header=False, index=False)
            
            if (i + 1) % 10 == 0:
                print(f"Generated {i + 1}/{num_samples} samples. Latest Triage Level: {triage_label}")
                
        except Exception as e:
            print(f"Error at sample {i+1}: {e}")
            continue
            
    if records:
        df_final_json = pd.DataFrame(records)[final_columns]
        df_final_json.to_json("manchester_postpartum_triage_v1.json", orient="records", indent=4)
        print("🎉 Final JSON & CSV datasets saved successfully with 4-class Manchester labels!")
        return df_final_json
    return pd.DataFrame()

# הרצה מלאה
df_final = build_dataset(num_samples=3000)

🚨 Starting generation of 3000 samples based on Manchester Triage Protocol...
Generated 10/3000 samples. Latest Triage Level: 0
Generated 20/3000 samples. Latest Triage Level: 2
Generated 30/3000 samples. Latest Triage Level: 1
Generated 40/3000 samples. Latest Triage Level: 0
Generated 50/3000 samples. Latest Triage Level: 1
Generated 60/3000 samples. Latest Triage Level: 2
Generated 70/3000 samples. Latest Triage Level: 2
Generated 80/3000 samples. Latest Triage Level: 1
Generated 90/3000 samples. Latest Triage Level: 2
Generated 100/3000 samples. Latest Triage Level: 1
Generated 110/3000 samples. Latest Triage Level: 2
Generated 120/3000 samples. Latest Triage Level: 0
Generated 130/3000 samples. Latest Triage Level: 3
Generated 140/3000 samples. Latest Triage Level: 3
Generated 150/3000 samples. Latest Triage Level: 3
Generated 160/3000 samples. Latest Triage Level: 0
Generated 170/3000 samples. Latest Triage Level: 1
Generated 180/3000 samples. Latest Triage Level: 2
Generated 190/